(searching)=

# Searching

```{contents}
:local:
:depth: 2
```

First-match linear search, iterative binary search, missing-target traces, and library return values are taught in {doc}`Chapter 15 <../15_arrays_linked_lists/1504_searching_sorting>`. This optional extension studies equality policies, duplicate boundaries, and repeated-query workloads.


## Equality and Ordering Policies

```{index} Equality and Ordering Policies
```

Choose a supplier-name policy explicitly. Ordinal equality is case-sensitive; ordinal case-insensitive equality can treat names with different case as one key. Use an ordering compatible with that equivalence when sorting for binary search. String comparisons need not be constant-time in character length.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] names={"Acme","BETA","acme","Delta"};
foreach(var policy in new[]{StringComparer.Ordinal,StringComparer.OrdinalIgnoreCase})
{
 int index=ReportSearch.LinearFirst(names,"ACME",policy,out long probes);
 Console.WriteLine($"first={index}, probes={probes}");
}
public static class ReportSearch
{
    public static int LinearFirst<T>(T[] values,T target,IEqualityComparer<T> equality,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(equality);
        probes=0;
        for(int i=0;i<values.Length;i++)
        {
            probes++;
            if(equality.Equals(values[i],target))return i;
        }
        return -1;
    }
    // Sorted under the same comparer. Returns any match, or -1.
    public static int BinaryAny<T>(T[] values,T target,IComparer<T> comparer,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order==0)return middle;
            if(order<0)low=middle+1;else high=middle;
        }
        return -1;
    }
    // First position >= target, or Length. Equality does not stop the loop.
    public static int LowerBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,false,out probes);
    // First position > target, or Length.
    public static int UpperBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,true,out probes);
    private static int Bound<T>(T[] values,T target,IComparer<T> comparer,bool upper,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order<0 || upper && order==0)low=middle+1;
            else high=middle;
        }
        return low;
    }
}


Comparison policy is part of the contract, not an incidental culture setting. For records, search the intended key; equality of whole records may be a different business rule. Binary searches below count calls to a three-way comparer, not each primitive relational operation inside that comparer.

## Lower and Upper Bounds for Duplicates

```{index} Lower and Upper Bounds for Duplicates
```

A lower bound is the first position with value ≥ target; an upper bound is the first position with value > target. Both can equal n. Lower-bound invariants: every index below low is < target, and every index at or above high is ≥ target. At low=high, that boundary is established. Upper bound replaces those relations with ≤ and >. Equality must keep searching rather than return early.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sorted={100,200,200,200,300};
foreach(int target in new[]{50,200,250,500})
{
 int lower=ReportSearch.LowerBound(sorted,target,Comparer<int>.Default,out long lp);
 int upper=ReportSearch.UpperBound(sorted,target,Comparer<int>.Default,out long up);
 bool found=lower<sorted.Length && sorted[lower]==target;
 Console.WriteLine($"target={target}, lower={lower}, upper={upper}, count={upper-lower}, first={(found?lower:-1)}, probes={lp+up}");
}
public static class ReportSearch
{
    public static int LinearFirst<T>(T[] values,T target,IEqualityComparer<T> equality,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(equality);
        probes=0;
        for(int i=0;i<values.Length;i++)
        {
            probes++;
            if(equality.Equals(values[i],target))return i;
        }
        return -1;
    }
    // Sorted under the same comparer. Returns any match, or -1.
    public static int BinaryAny<T>(T[] values,T target,IComparer<T> comparer,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order==0)return middle;
            if(order<0)low=middle+1;else high=middle;
        }
        return -1;
    }
    // First position >= target, or Length. Equality does not stop the loop.
    public static int LowerBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,false,out probes);
    // First position > target, or Length.
    public static int UpperBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,true,out probes);
    private static int Bound<T>(T[] values,T target,IComparer<T> comparer,bool upper,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order<0 || upper && order==0)low=middle+1;
            else high=middle;
        }
        return low;
    }
}


Equal records occupy [lower,upper), so upper−lower counts occurrences and lower identifies the first sorted-order match. Two boundary searches cost Θ(log n) worst-case probes; enumerating k matching records adds Θ(k) output work. The original arrival position cannot be recovered from a sorted amount-only array: retain record IDs or original indexes.

## Repeated-Query Workloads

```{index} searching; workload costs
```


For q lookups, scanning costs up to Θ(qn). Sorting once and searching costs Θ(n log n+q log n), assuming a comparison sort and static data. Updates can invalidate the sorted snapshot or require maintenance. Sorting merely to answer one query may add work; an existing sorted array can support binary search even for one query. Dictionary exact-key lookup needs its own duplicate and update policy; ordered ranges and all matches are different requirements.

## Exercise

```{index} Exercise
```

Return the full sorted match range for an invoice amount, including no-match and empty cases. Verify lower/upper boundaries before indexing.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sorted={100,200,200,300};
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[] sorted={100,200,200,300};
foreach(int target in new[]{200,250})
{
 int lower=ReportSearch.LowerBound(sorted,target,Comparer<int>.Default,out _);
 int upper=ReportSearch.UpperBound(sorted,target,Comparer<int>.Default,out _);
 Console.WriteLine($"target={target}, range=[{lower},{upper}), count={upper-lower}");
}
Console.WriteLine($"empty lower={ReportSearch.LowerBound(Array.Empty<int>(),200,Comparer<int>.Default,out _)}");
public static class ReportSearch
{
    public static int LinearFirst<T>(T[] values,T target,IEqualityComparer<T> equality,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(equality);
        probes=0;
        for(int i=0;i<values.Length;i++)
        {
            probes++;
            if(equality.Equals(values[i],target))return i;
        }
        return -1;
    }
    // Sorted under the same comparer. Returns any match, or -1.
    public static int BinaryAny<T>(T[] values,T target,IComparer<T> comparer,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order==0)return middle;
            if(order<0)low=middle+1;else high=middle;
        }
        return -1;
    }
    // First position >= target, or Length. Equality does not stop the loop.
    public static int LowerBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,false,out probes);
    // First position > target, or Length.
    public static int UpperBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,true,out probes);
    private static int Bound<T>(T[] values,T target,IComparer<T> comparer,bool upper,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order<0 || upper && order==0)low=middle+1;
            else high=middle;
        }
        return low;
    }
}


target=200, range=[1,3), count=2
target=250, range=[3,3), count=0
empty lower=0


```{rubric} Footnotes
```
[^1]: [Microsoft’s Array.BinarySearch contract](https://learn.microsoft.com/en-us/dotnet/api/system.array.binarysearch?view=net-10.0) specifies negative insertion-position encoding and sorted-input requirements.
[^2]: Probe counts assume constant-cost access and comparison; variable-length keys need an additional key-length cost model.